# MORPH ? Part 2: map variation onto a structure

This notebook retrieves the public prepared dataset, validates chain A against UniProt residue numbering, maps gnomAD variation onto a PDB, and downloads the annotated result.

Use a Google Colab CPU runtime. You may upload a PDB from `1-Boltz2.ipynb` or provide another structure that meets the validation requirements.


#**⚠️ Before you continue: change runtime from GPU to CPU (no need to waste ressources 🔥💸 or 🌳)**

In Google Colab:

1.   Click Runtime → Change runtime type
2.   Set Hardware accelerator to CPU
3.   **Click Save**

#**⚠️ to map allele frequency to .pdb 🧭🗺️📍**

Requires a pdb meeting at least these **`requirements`**:

1.   whose **`chain A`** sequence matches **`UniProtKB reviewed (Swiss-Prot)`** & **`Homo sapiens (Human)`** entry
2.   has **contiguous sequence** (**i.e no gaps**)
3.   has available corresponding **`gnomAD allele frequency`** data




#**🧱 Let’s start by installing the required environment for MORPH**

The [MORPH code](https://github.com/lucaschestnuttree/MORPH) and [variant data](https://zenodo.org/records/18163874) are public. No GitHub or Zenodo credentials are needed for this part of the notebook.


In [ ]:

# @title 🧱 Automated MORPH Dependencies Install & Restart (Session will crash)

import os
import sys
import signal
import time
import subprocess
import requests
from pathlib import Path

# ==================================================
# CONFIGURATION
# ==================================================

PYMUT_REPO = "https://github.com/gerdos/pyMUT"

MORPH_REPO = "lucaschestnuttree/MORPH"
REF = "main"
RAW_BASE = f"https://raw.githubusercontent.com/{MORPH_REPO}/{REF}"

FILES = [
    (
        "src/2-map_2_structure/pymut_gnomad_af_2_structure.py",
        "src/2-mapping/map_2_structure/pymut_gnomad_af_2_structure.py",
    ),
    (
        "src/2-map_2_structure/pymut_gnomad_anc_2_structure.py",
        "src/2-mapping/map_2_structure/pymut_gnomad_anc_2_structure.py",
    ),
]

DEST_DIR = Path("pyMUT/src/2-mapping/map_2_structure")

NUMPY_VER = "1.26.4"
PANDAS_VER = "2.1.4"

def run(cmd):
    subprocess.run(cmd, check=True)

print("=" * 55)
print("🧱 MORPH AUTOMATED INSTALLER")
print("Target: Google Colab 2026.07 | CPU")
print("=" * 55, flush=True)

# ==================================================
# STEP 1: INSTALL COMPATIBLE DEPENDENCIES
# ==================================================

print("\n[1/4] Installing dependencies...", flush=True)

# Do not uninstall or force-reinstall packages.
# This avoids Debian-managed package RECORD errors.
# pip keeps compatible packages already installed.

run([
    sys.executable, "-m", "pip", "install",
    "--prefer-binary",
    f"numpy=={NUMPY_VER}",
    f"pandas=={PANDAS_VER}",
    "biopython>=1.7",
    "requests"
])

print("\n✅ Dependency installation completed.")

# Verify in a fresh Python process
check_code = """
import numpy, pandas, Bio
print('NumPy:', numpy.__version__)
print('Pandas:', pandas.__version__)
print('Biopython:', Bio.__version__)
assert numpy.__version__ == '1.26.4'
assert pandas.__version__ == '2.1.4'
"""

run([sys.executable, "-c", check_code])

# ==================================================
# STEP 2: CLONE pyMUT
# ==================================================

print("\n[2/4] Preparing pyMUT...", flush=True)

if not Path("pyMUT").exists():
    run([
        "git", "clone", "--depth", "1",
        PYMUT_REPO, "pyMUT"
    ])
    print("✅ pyMUT cloned.")
else:
    print("✅ pyMUT directory already exists.")

# ==================================================
# STEP 3: DOWNLOAD MORPH SCRIPTS
# ==================================================

print("\n[3/4] Downloading MORPH scripts...", flush=True)

DEST_DIR.mkdir(parents=True, exist_ok=True)

def download_public_script(repo_paths):
    for repo_path in repo_paths:
        url = f"{RAW_BASE}/{repo_path}"

        response = requests.get(url, timeout=60)

        if response.status_code == 404:
            continue

        response.raise_for_status()

        output = DEST_DIR / Path(repo_path).name
        output.write_bytes(response.content)

        print(f"✅ Downloaded: {output}")
        return

    raise FileNotFoundError(
        f"MORPH script not found: {repo_paths}"
    )

for paths in FILES:
    download_public_script(paths)

print("\n✅ MORPH scripts ready.")

# ==================================================
# STEP 4: AUTOMATIC RUNTIME RESTART
# ==================================================

print("\n[4/4] Restarting Python runtime...", flush=True)

print("\n" + "=" * 55)
print("✅ MORPH INSTALLATION COMPLETE!")
print("📦 Dependencies installed and verified.")
print("📂 pyMUT and MORPH scripts prepared.")
print("")
print("🔄 Automatically restarting Colab...")
print("⏳ Wait for the session to reconnect.")
print("▶️ Then continue to the next notebook cell.")
print("=" * 55, flush=True)

sys.stdout.flush()
sys.stderr.flush()
time.sleep(3)

# Restart Python while retaining installed packages
os.kill(os.getpid(), signal.SIGKILL)


In [ ]:
# @title 🔌 After restart: make pyMUT importable
import os, sys
from pathlib import Path

PYMUT_DIR = Path("pyMUT").resolve()

# Notebook imports
if str(PYMUT_DIR) not in sys.path:
    sys.path.insert(0, str(PYMUT_DIR))

# Subprocess imports
os.environ["PYTHONPATH"] = str(PYMUT_DIR) + (":" + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else "")

import numpy as np
print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("PYTHONPATH:", os.environ["PYTHONPATH"])

import pymut
print("✅ pymut imported from:", pymut.__file__)


In [ ]:
#@title #🔍 Retrieve necessary lookup table
# =======================
# Zenodo record downloader (public access)
# =======================

import os
import time
import requests

# ---- CONFIG ----
RECORD_ID = "18163874"
FILENAME = "lookup_amissenseXgnomAD.tsv"
OUT_DIR = "zenodo_downloads"

# ---- DOWNLOAD ----
os.makedirs(OUT_DIR, exist_ok=True)
dest = os.path.join(OUT_DIR, FILENAME)

if os.path.exists(dest) and os.path.getsize(dest) > 0:
    print(f"✅ Using cached file: {dest}")
else:
    url = f"https://zenodo.org/records/{RECORD_ID}/files/{FILENAME}?download=1"

    max_retries = 6
    timeout_s = 300

    for attempt in range(1, max_retries + 1):
        try:
            print(f"⬇️ Download attempt {attempt}/{max_retries}")
            with requests.get(url, stream=True, timeout=timeout_s) as r:
                if r.status_code in (429, 500, 502, 503, 504):
                    raise requests.HTTPError(f"HTTP {r.status_code}", response=r)
                r.raise_for_status()

                tmp = dest + ".part"
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):
                        if chunk:
                            f.write(chunk)

                os.replace(tmp, dest)
                print(f"✅ Downloaded successfully: {dest}")
                break

        except Exception as e:
            # cleanup partial
            try:
                if os.path.exists(dest + ".part"):
                    os.remove(dest + ".part")
            except Exception:
                pass

            if attempt == max_retries:
                raise RuntimeError(f"❌ Download failed after {max_retries} attempts") from e

            wait = 2 * (2 ** (attempt - 1))
            print(f"⚠️ Error: {e}")
            print(f"⏳ Retrying in {wait}s…")
            time.sleep(wait)


#**☁️ Upload a suitable .pdb**

*   If you followed **`part I`**  you can upload the file you Downloaded
*   Otherwise a file that meets **`requirements`**

We will verify that this file meets our requirements




In [ ]:
#@title ✅ Validate Selected PDB Input (UniProt ID + Select PDB) + Fetch & Unpack Zenodo tar.gz (after all tests pass)
#@markdown Mandatory inputs:
#@markdown - **UniProt ID**
#@markdown - **Click ▶️ Run** to open a **file picker in output cell** and select a **`.pdb`** file from your computer
#@markdown
#@markdown This cell:
#@markdown 1) Ensures local **gnomAD lookup TSV** exists (download from the public Zenodo record if missing)
#@markdown 2) Validates **UniProt entry (reviewed Swiss-Prot, Homo sapiens)**
#@markdown 3) Extracts chain A from ATOM CA coords
#@markdown    - FAILS if **chain A has internal residue numbering gaps** (prints gapped sequence with '-' at gaps)
#@markdown    - Requires **100% identity** vs UniProt for residues present (allows missing N/C termini)
#@markdown 4) Confirms available **gnomAD allele frequency data** (lookup row exists)
#@markdown 5) If all pass: downloads the corresponding **.tar.gz** from the same Zenodo record and safely unpacks it

uniprot_id = "Q9Y2D9"  #@param {type:"string"}

import os, time
from pathlib import Path

import requests
import pandas as pd
import tarfile

# ==========================================================
# CONFIG
# ==========================================================
# UniProt REST
UNIPROT_ENTRY_URL = "https://rest.uniprot.org/uniprotkb/{accession}?format=json"

# Zenodo record + lookup file
RECORD_ID = "18163874"
LOOKUP_FILENAME = "lookup_amissenseXgnomAD.tsv"

# Storage locations
OUT_DIR = Path("zenodo_downloads")
LOOKUP_PATH = OUT_DIR / LOOKUP_FILENAME

TAR_DIR = OUT_DIR / "tars"
EXTRACT_DIR = OUT_DIR / "extracted"

OUT_DIR.mkdir(parents=True, exist_ok=True)
TAR_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Globals for downstream steps
current_uniprot_id = None
current_sequence = None            # canonical: UniProt
current_sequence_uniprot = None
current_sequence_pdb = None        # chain A from ATOM coords (UNGAPPED)
current_pdb_path = None
has_gnomad_lookup = None
gnomad_lookup_path = None
current_tar_info = None

# ==========================================================
# ZENODO DOWNLOAD (retry/backoff + atomic write)
# ==========================================================
def download_zenodo_file(record_id: str, filename: str, out_path: Path,
                         max_retries: int = 6, timeout_s: int = 300,
                         verbose: bool = False) -> Path:
    url = f"https://zenodo.org/records/{record_id}/files/{filename}?download=1"

    if out_path.exists() and out_path.stat().st_size > 0:
        if verbose:
            print(f"Using cached file: {out_path}")
        return out_path

    tmp = out_path.with_suffix(out_path.suffix + ".part")

    for attempt in range(1, max_retries + 1):
        try:
            if verbose:
                print(f"Downloading {filename} (attempt {attempt}/{max_retries})")

            with requests.get(url, stream=True, timeout=timeout_s) as r:
                if r.status_code in (429, 500, 502, 503, 504):
                    raise requests.HTTPError(f"HTTP {r.status_code}", response=r)
                r.raise_for_status()

                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):
                        if chunk:
                            f.write(chunk)

            os.replace(tmp, out_path)
            return out_path

        except Exception:
            if tmp.exists():
                tmp.unlink()
            if attempt == max_retries:
                raise
            time.sleep(2 * (2 ** (attempt - 1)))

# ==========================================================
# ✅ LOOKUP TABLE ENSURER (this fixes the NameError)
# ==========================================================
def ensure_lookup_table() -> Path:
    """
    Ensure the gnomAD lookup TSV exists locally.
    - If already present and non-empty: return it
    - Else: download from the public Zenodo record
    Also performs a light schema check (required columns).
    """
    # If an earlier cell already created some path/variable, try to reuse it safely.
    for candidate_name in ("lookup_path", "gnomad_lookup_path", "LOOKUP_PATH"):
        if candidate_name in globals():
            val = globals()[candidate_name]
            try:
                p = Path(val)
                if p.exists() and p.stat().st_size > 0 and p.name.endswith(".tsv"):
                    # quick column check
                    df_head = pd.read_csv(p, sep="\t", nrows=5, dtype=str)
                    required = {"uniprot_id", "tar_file", "tsv_file"}
                    missing = required - set(df_head.columns)
                    if missing:
                        raise ValueError(
                            f"Lookup TSV exists but is missing required columns: {sorted(missing)}\n"
                            f"Found columns: {list(df_head.columns)}"
                        )
                    return p
            except Exception:
                pass  # fall back to standard LOOKUP_PATH below

    # Standard location
    out_path = LOOKUP_PATH

    if out_path.exists() and out_path.stat().st_size > 0:
        df_head = pd.read_csv(out_path, sep="\t", nrows=5, dtype=str)
        required = {"uniprot_id", "tar_file", "tsv_file"}
        missing = required - set(df_head.columns)
        if missing:
            raise ValueError(
                f"Lookup TSV exists but is missing required columns: {sorted(missing)}\n"
                f"Found columns: {list(df_head.columns)}"
            )
        return out_path

    # Need to download
    downloaded = download_zenodo_file(RECORD_ID, LOOKUP_FILENAME, out_path, verbose=True)

    # Validate schema after download
    df_head = pd.read_csv(downloaded, sep="\t", nrows=5, dtype=str)
    required = {"uniprot_id", "tar_file", "tsv_file"}
    missing = required - set(df_head.columns)
    if missing:
        raise ValueError(
            f"Downloaded lookup TSV is missing required columns: {sorted(missing)}\n"
            f"Found columns: {list(df_head.columns)}"
        )

    return downloaded

# ==========================================================
# SAFE TAR EXTRACTION
# ==========================================================
def _is_within_directory(base: Path, target: Path) -> bool:
    base = base.resolve()
    target = target.resolve()
    try:
        target.relative_to(base)
        return True
    except ValueError:
        return False

def safe_extract_tar_gz(tar_path: Path, out_dir: Path) -> None:
    """Safely extract tar.gz into out_dir (blocks path traversal, future-proof)."""
    out_dir.mkdir(parents=True, exist_ok=True)

    with tarfile.open(tar_path, "r:gz") as tf:
        # Explicit path traversal check
        for member in tf.getmembers():
            member_path = out_dir / member.name
            if not _is_within_directory(out_dir, member_path):
                raise RuntimeError(f"Unsafe tar member path: {member.name}")

        # Python 3.14+ safe extraction (fallback for older versions)
        try:
            tf.extractall(out_dir, filter="data")
        except TypeError:
            tf.extractall(out_dir)

def download_and_unpack_uniprot_tar(uniprot_id: str, lookup_tsv: Path) -> Path:
    df = pd.read_csv(lookup_tsv, sep="\t", dtype=str)

    hit = df[df["uniprot_id"].astype(str).str.strip() == uniprot_id]
    if hit.empty:
        raise ValueError(f"No gnomAD data for UniProt ID {uniprot_id}")

    row = hit.iloc[0]
    tar_file = row["tar_file"].strip()
    expected_tsv = row["tsv_file"].strip()

    tar_path = TAR_DIR / tar_file
    download_zenodo_file(RECORD_ID, tar_file, tar_path, verbose=False)

    stem = tar_file.replace(".tar.gz", "")
    out_dir = EXTRACT_DIR / stem

    if not (out_dir.exists() and any(out_dir.iterdir())):
        safe_extract_tar_gz(tar_path, out_dir)

    matches = list(out_dir.rglob(expected_tsv))
    if not matches:
        raise RuntimeError(f"Expected TSV not found: {expected_tsv}")

    return matches[0]

# ==========================================================
# PDB PICKER (Colab)
# ==========================================================
def select_pdb_via_picker() -> str:
    try:
        from google.colab import files
    except Exception:
        raise RuntimeError(
            "File picker requires Google Colab (google.colab.files). "
            "Please run this notebook in Colab."
        )

    print("📂 Please select a .pdb file from your machine...")
    uploaded = files.upload()

    if not uploaded:
        raise ValueError("No file selected. A .pdb file is required.")

    uploaded_name = next(iter(uploaded.keys()))
    if not uploaded_name.lower().endswith(".pdb"):
        raise ValueError(f"Selected file '{uploaded_name}' is not a .pdb")

    canonical = "input.pdb"

    # Overwrite canonical file
    if os.path.exists(canonical):
        os.remove(canonical)

    # Move uploaded file to canonical name
    os.replace(uploaded_name, canonical)

    # Optional cleanup of extra pdbs
    for fn in os.listdir("."):
        if fn != canonical and fn.lower().endswith(".pdb"):
            try:
                os.remove(fn)
            except Exception:
                pass

    return canonical

# ==========================================================
# UNIPROT HELPERS
# ==========================================================
def fetch_uniprot_entry(accession: str) -> dict:
    accession = accession.strip()
    if not accession:
        raise ValueError("UniProt ID is empty.")
    url = UNIPROT_ENTRY_URL.format(accession=accession)
    r = requests.get(url, timeout=60)
    if r.status_code == 404:
        raise ValueError(f"No UniProtKB entry found for '{accession}'.")
    if not r.ok:
        raise ValueError(f"UniProt API error (HTTP {r.status_code}).")
    return r.json()

def validate_entry(entry: dict) -> tuple[bool, str]:
    entry_type = entry.get("entryType", "")
    sci_name = entry.get("organism", {}).get("scientificName", "")

    is_reviewed = entry_type == "UniProtKB reviewed (Swiss-Prot)"
    is_human = "Homo sapiens" in sci_name

    lines = []
    lines.append("✓ UniProtKB reviewed (Swiss-Prot)" if is_reviewed else f"✗ entryType: {entry_type}")
    lines.append("✓ Organism: Homo sapiens (Human)" if is_human else f"✗ Organism: {sci_name}")
    return (is_reviewed and is_human), "\n".join(lines)

def extract_uniprot_sequence(entry: dict) -> str:
    seq = entry.get("sequence", {}).get("value")
    if not seq:
        raise ValueError("No sequence found in UniProt entry.")
    return seq.replace("\n", "").strip()

# ==========================================================
# PDB HELPERS (chain A from ATOM CA coords + gap detection)
# ==========================================================
_AA3_TO_1 = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G","HIS":"H",
    "ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S","THR":"T","TRP":"W",
    "TYR":"Y","VAL":"V","SEC":"U","PYL":"O",
}

def pdb_basic_sanity(pdb_text: str) -> None:
    if "ATOM" not in pdb_text:
        raise ValueError("PDB has no ATOM records; cannot extract sequence from coordinates.")

def _parse_atom_line(line: str):
    atom_name = line[12:16].strip()
    resname = line[17:20].strip()
    chain = line[21].strip() or "_"
    resseq_str = line[22:26].strip()
    icode = line[26].strip() or ""
    if not resseq_str:
        raise ValueError("Malformed PDB ATOM line: missing residue number.")
    return atom_name, resname, chain, int(resseq_str), icode

def extract_chainA_sequence_from_coords_with_gapinfo(pdb_text: str):
    """
    Extract chain A sequence from CA atoms.
    Also constructs a gapped-by-numbering sequence (inserting '-' for numbering jumps)
    so we can print where gaps are, then FAIL EARLY if gaps exist.
    """
    residues = []
    seen = set()
    has_chain_a = False

    for ln in pdb_text.splitlines():
        if not ln.startswith("ATOM"):
            continue
        atom_name, resname, chain, resseq, icode = _parse_atom_line(ln)
        if chain != "A":
            continue
        has_chain_a = True
        if atom_name != "CA":
            continue
        key = (resseq, icode)
        if key in seen:
            continue
        seen.add(key)
        residues.append((resseq, icode, resname))

    if not has_chain_a:
        raise ValueError("No chain A found in ATOM records.")
    if not residues:
        raise ValueError("Chain A found, but no CA atoms in ATOM records; cannot build residue sequence.")

    residues.sort(key=lambda x: (x[0], x[1]))

    seq_ungapped = "".join(_AA3_TO_1.get(resname.upper(), "X") for (_, _, resname) in residues)

    gapped = []
    gaps = []
    prev_resseq = residues[0][0]
    gapped.append(_AA3_TO_1.get(residues[0][2].upper(), "X"))

    for (resseq, icode, resname) in residues[1:]:
        jump = resseq - prev_resseq
        if jump > 1:
            gapped.extend(["-"] * (jump - 1))
            gaps.append((prev_resseq + 1, resseq - 1))
        gapped.append(_AA3_TO_1.get(resname.upper(), "X"))
        prev_resseq = resseq

    seq_gapped = "".join(gapped)
    return seq_ungapped, seq_gapped, gaps, residues

def require_chain_numbering_matches_uniprot(residues: list, uniprot_seq: str) -> tuple[int, int]:
    """Check every chain A CA residue against the same numbered UniProt position."""
    first_resseq = residues[0][0]
    if first_resseq < 1:
        raise ValueError(f"Chain A starts at invalid PDB residue number {first_resseq}.")
    for index, (resseq, icode, resname) in enumerate(residues):
        expected_resseq = first_resseq + index
        if icode or resseq != expected_resseq:
            raise ValueError(
                f"Chain A numbering is not consecutive at PDB residue {resseq}{icode}. "
                "Insertion codes and internal gaps are not supported by the mapper."
            )
        if resseq > len(uniprot_seq):
            raise ValueError(f"PDB residue {resseq} is beyond UniProt length {len(uniprot_seq)}.")
        pdb_aa = _AA3_TO_1.get(resname.upper())
        if pdb_aa is None:
            raise ValueError(f"Unsupported chain A residue {resname} at PDB position {resseq}.")
        uniprot_aa = uniprot_seq[resseq - 1]
        if pdb_aa != uniprot_aa:
            raise ValueError(
                f"Chain A residue numbering does not match UniProt at position {resseq}: "
                f"PDB has {pdb_aa}, UniProt has {uniprot_aa}. "
                "Keep canonical UniProt residue numbers when using a truncated PDB."
            )
    chain_seq = "".join(_AA3_TO_1[resname.upper()] for _, _, resname in residues)
    first_match = uniprot_seq.find(chain_seq)
    if uniprot_seq.find(chain_seq, first_match + 1) != -1:
        raise ValueError(
            "Chain A sequence occurs at multiple UniProt positions, so its numbering cannot "
            "be verified from sequence alone. Use a longer structure or independent numbering evidence."
        )
    return first_resseq - 1, residues[-1][0]

# ==========================================================
# LOOKUP HELPERS
# ==========================================================
def lookup_row_for_uniprot(tsv_path: Path, uniprot_id: str) -> dict | None:
    df = pd.read_csv(tsv_path, sep="\t", dtype=str, low_memory=False)
    if "uniprot_id" not in df.columns:
        raise ValueError(f"Lookup TSV does not contain 'uniprot_id' column. Found: {list(df.columns)}")
    hit = df[df["uniprot_id"].astype(str).str.strip() == uniprot_id]
    if hit.empty:
        return None
    return hit.iloc[0].to_dict()

# ==========================================================
# RUN
# ==========================================================
try:
    uniprot_id = uniprot_id.strip()
    if not uniprot_id:
        raise ValueError("UniProt ID is required.")

    # 0) Ensure lookup table exists (download if needed)
    lookup_path = ensure_lookup_table()
    gnomad_lookup_path = str(lookup_path)

    # 1) Select PDB
    pdb_file = select_pdb_via_picker()
    current_pdb_path = pdb_file

    with open(pdb_file, "r", encoding="utf-8", errors="replace") as f:
        pdb_text = f.read()
    pdb_basic_sanity(pdb_text)

    # 2) Chain A extraction + FAIL EARLY on internal numbering gaps
    seq_chainA, seq_chainA_gapped, chainA_gaps, residues_chainA = extract_chainA_sequence_from_coords_with_gapinfo(pdb_text)
    current_sequence_pdb = seq_chainA

    print(f"✅ PDB loaded: {pdb_file}")
    print(f"✅ Chain A extracted from ATOM CA coords. Ungapped length: {len(seq_chainA)} aa")

    if chainA_gaps:
        gap_str = ", ".join([f"{a}-{b}" if a != b else f"{a}" for a, b in chainA_gaps])
        print(f"\n❌ FAIL: Chain A has INTERNAL residue numbering gap(s): {gap_str}")
        print("\nChain A sequence with '-' inserted for numbering gaps:")
        print(seq_chainA_gapped)
        raise ValueError("Stopping because PDB chain A has internal gaps.")
    else:
        print("✓ No internal residue numbering gaps detected in chain A.")
        print("\nChain A sequence (full, ungapped):")
        print(seq_chainA)
        print()

    # 3) UniProt validation
    entry = fetch_uniprot_entry(uniprot_id)
    ok, msg = validate_entry(entry)

    print(f"Checking UniProt entry for {uniprot_id}\n")
    print(msg)
    print()

    if not ok:
        raise ValueError("Stopping because UniProt criteria failed.")

    seq_uni = extract_uniprot_sequence(entry)
    current_sequence_uniprot = seq_uni
    current_uniprot_id = uniprot_id
    current_sequence = seq_uni  # canonical

    print(f"✅ Accepted. UniProt sequence length: {len(seq_uni)} aa")

    # 3b) Verify identity and canonical residue numbering before mapping
    start0, end0 = require_chain_numbering_matches_uniprot(residues_chainA, seq_uni)
    print("Chain A sequence and residue numbers match UniProt (allows N/C-terminal truncation).")
    print(f"Chain A maps to UniProt positions (1-based): {start0 + 1}-{end0}")
    print(f"N-terminal truncation vs UniProt: {start0} aa")
    print(f"C-terminal truncation vs UniProt: {len(seq_uni) - end0} aa")
    print()

    # 4) Confirm gnomAD allele frequency data exists (lookup row exists)
    row = lookup_row_for_uniprot(lookup_path, uniprot_id)
    has_gnomad_lookup = row is not None

    print(f"🗂️ gnomAD lookup table: {lookup_path}")
    print("gnomAD allele frequency availability for this UniProt ID:")
    if has_gnomad_lookup:
        print("✅ AVAILABLE")
        print(f"Found: tar_file={row.get('tar_file')}  tsv_file={row.get('tsv_file')}")
    else:
        print("❌ NOT FOUND")
        raise ValueError("Stopping because no gnomAD allele frequency data was found for this UniProt ID.")

    # 5) If all tests passed: download/unpack and print only success message
    gnomad_tsv_path = download_and_unpack_uniprot_tar(uniprot_id, lookup_path)
    print(f"✅ Retrieved gnomAD allele frequency data for {uniprot_id}")
    # (optional) print where it landed:
    # print(f"📄 TSV path: {gnomad_tsv_path}")

except Exception as e:
    print("❌ Error:", e)


#**🏗️ Build model with allele frequency and alternate sidechains in .pdb 🧭🗺️📍**

In [ ]:
#@title Run gnomAD structure mapping (≈30 min for ~500 aa) ☕🍵
#@markdown ---
#@markdown ### Mapping mode
#@markdown This step maps **gnomAD missense variation** onto the protein structure.
#@markdown
#@markdown **Allele frequency**
#@markdown - Uses *global* allele frequency across all individuals
#@markdown
#@markdown **Ancestry frequency**
#@markdown - Uses *population- or sex-specific* allele frequencies
#@markdown
#@markdown ---
#@markdown ### Allowed gnomAD categories
#@markdown **Population prefixes**
#@markdown - `nfe` — Non-Finnish European
#@markdown - `fin` — Finnish
#@markdown - `mid` — Middle Eastern
#@markdown - `remaining` — Remaining / unassigned populations
#@markdown - `ami` — Amish
#@markdown - `asj` — Ashkenazi Jewish
#@markdown - `afr` — African / African American
#@markdown - `eas` — East Asian
#@markdown - `sas` — South Asian
#@markdown - `amr` — Admixed American
#@markdown
#@markdown **Biological sex prefixes**
#@markdown - `XX` — Biological females
#@markdown - `XY` — Biological males
#@markdown
#@markdown ⚠️ **Important constraint**
#@markdown - A maximum of **7 categories** can be used at once
#@markdown - Categories are mapped to chains **T, U, V, W, X, Y, Z**
#@markdown - The run will **fail immediately** if more than 7 are provided
#@markdown ---

import os, sys, subprocess, time
from pathlib import Path

# -------------------------
# Parameters
# -------------------------
mapping = "ancestry frequency"  #@param ["allele frequency", "ancestry frequency"]
CATEGORIES = "afr,nfe,sas,eas,fin"  #@param {type:"string"}

# -------------------------
# Resolve inputs from previous cells
# -------------------------
def _as_path(x):
    if x is None:
        return None
    return x if isinstance(x, Path) else Path(str(x))

tsv_path = _as_path(globals().get("gnomad_tsv_path"))
pdb_path = _as_path(globals().get("current_pdb_path"))

if tsv_path is None or pdb_path is None:
    raise RuntimeError(
        "Missing inputs.\n"
        "Please run the previous validation/download cell (Part II) before this step."
    )

base_out = Path("morph_outputs").resolve()

if mapping == "allele frequency":
    out_dir = base_out / "allele_frequency"
else:
    out_dir = base_out / "ancestry_frequency"

out_dir.mkdir(parents=True, exist_ok=True)


# -------------------------
# Enforce mode-specific logic
# -------------------------
if mapping == "allele frequency":
    # Hard-disable categories in allele-frequency mode
    CATEGORIES = None
    SCRIPT = Path(
        "pyMUT/src/2-mapping/map_2_structure/pymut_gnomad_af_2_structure.py"
    ).resolve()

elif mapping == "ancestry frequency":
    SCRIPT = Path(
        "pyMUT/src/2-mapping/map_2_structure/pymut_gnomad_anc_2_structure.py"
    ).resolve()

    cats = [c.strip() for c in CATEGORIES.split(",") if c.strip()]

    if not cats:
        raise ValueError(
            "Ancestry frequency mapping requires at least one gnomAD category."
        )

    if len(cats) > 7:
        raise ValueError(
            f"You specified {len(cats)} categories, but the maximum allowed is 7.\n"
            "This limit exists because categories are mapped to chains T–Z.\n"
            "Please reduce the list and try again."
        )

else:
    raise ValueError(f"Unknown mapping mode: {mapping}")

# -------------------------
# Sanity checks
# -------------------------
def _must_exist(p: Path, label: str):
    if not p.exists():
        raise FileNotFoundError(f"{label} not found: {p}")

_must_exist(SCRIPT, "SCRIPT")
_must_exist(tsv_path, "TSV")
_must_exist(pdb_path, "PDB")

# -------------------------
# Ensure subprocess can import pyMUT
# -------------------------
env = os.environ.copy()
pymut_dir = str(Path("pyMUT").resolve())
existing = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = ":".join(dict.fromkeys([pymut_dir] + existing.split(":")))

# -------------------------
# Build command
# -------------------------
if mapping == "allele frequency":
    cmd = [sys.executable, "-u", str(SCRIPT), str(tsv_path), str(pdb_path), str(out_dir)]
else:
    cmd = [
        sys.executable,
        "-u",
        str(SCRIPT),
        str(tsv_path),
        str(pdb_path),
        str(out_dir),
        CATEGORIES,
    ]

print("Running:", " ".join(cmd))

# -------------------------
# Run with live output
# -------------------------
TIMEOUT_SECONDS = 3600
start = time.time()

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

try:
    for line in iter(proc.stdout.readline, ""):
        print(line, end="")
        if time.time() - start > TIMEOUT_SECONDS:
            proc.kill()
            raise TimeoutError("Process exceeded time limit and was terminated.")
    proc.stdout.close()
    rc = proc.wait()
finally:
    try:
        proc.kill()
    except Exception:
        pass

if rc != 0:
    raise RuntimeError(f"Script failed with return code {rc}")

print("\n✅ gnomAD mapping completed successfully.")


In [ ]:
#@title Zip and download mapping outputs
#@markdown This will package and download the outputs produced by the selected mapping mode.

from pathlib import Path
import shutil

# Base output directory (must match the run cell)
base_out = Path("morph_outputs").resolve()

# Select subdirectory based on mapping mode
if mapping == "allele frequency":
    out_dir = base_out / "allele_frequency"
    zip_name = "morph_allele_frequency_outputs"
elif mapping == "ancestry frequency":
    out_dir = base_out / "ancestry_frequency"
    zip_name = "morph_ancestry_frequency_outputs"
else:
    raise ValueError(f"Unknown mapping mode: {mapping}")

# Sanity check
if not out_dir.exists():
    raise FileNotFoundError(
        f"Expected output directory not found:\n  {out_dir}\n"
        "Please run the mapping cell first."
    )

zip_path = out_dir.parent / f"{zip_name}.zip"

# Remove old archive if present
if zip_path.exists():
    zip_path.unlink()

# Create archive
shutil.make_archive(
    base_name=str(zip_path.with_suffix("")),
    format="zip",
    root_dir=out_dir.parent,
    base_dir=out_dir.name,
)

print(f"Created archive: {zip_path}")

# Download (Colab)
from google.colab import files
files.download(str(zip_path))
